In [37]:
import pandas as pd
import numpy as np

In [38]:
ISOLAR_PATH = "/home/yiran/developments/Isolar.txt"
R_PATH = "/home/yiran/developments/R.txt"
separator = "\t"

In [39]:
# def calculate(isolar_path, r_path, lower = 300, upper = 2400, separator='\t'):

In [40]:
isolar_path = ISOLAR_PATH
r_path = R_PATH
lower = 300
upper = 2500
separator = "\t"

In [43]:
import pandas as pd
import numpy as np

# Read the data files
isolar = pd.read_csv(
    isolar_path, sep=separator, header=None, names=["wavelength_nm", "I_solar"]
)
r = pd.read_csv(r_path, sep=separator, header=None, names=["wavelength_nm", "R"])

# Ensure data is numeric and drop any rows with text/missing values (e.g., headers)
isolar["wavelength_nm"] = pd.to_numeric(isolar["wavelength_nm"], errors='coerce')
isolar["I_solar"] = pd.to_numeric(isolar["I_solar"], errors='coerce')
r["wavelength_nm"] = pd.to_numeric(r["wavelength_nm"], errors='coerce')
r["R"] = pd.to_numeric(r["R"], errors='coerce')

isolar = isolar.dropna().sort_values("wavelength_nm")
r = r.dropna().sort_values("wavelength_nm")

# Determine the valid overlapping range between the two datasets and the user bounds
overlap_min = max(isolar["wavelength_nm"].min(), r["wavelength_nm"].min(), lower)
overlap_max = min(isolar["wavelength_nm"].max(), r["wavelength_nm"].max(), upper)

# Filter the solar spectrum to this common range
isolar_valid = isolar[
    (isolar["wavelength_nm"] >= overlap_min) & (isolar["wavelength_nm"] <= overlap_max)
].copy()

# Convert wavelengths from nm to um to standardise units
isolar_valid["wavelength_um"] = isolar_valid["wavelength_nm"] / 1000
wl_common_um = isolar_valid["wavelength_um"].values
I_solar_common = isolar_valid["I_solar"].values

r_wl_um = r["wavelength_nm"].values / 1000
r_val = r["R"].values / 100.0  # Convert reflectance from percentage to a fraction (0 to 1)

# Interpolate the reflectance data onto the solar spectrum's wavelength grid
# This is the scientific approach to harmonise mismatched data points
R_interp = np.interp(wl_common_um, r_wl_um, r_val)

# Clip the interpolated reflectance to physical limits to prevent anomalies
R_interp = np.clip(R_interp, 0.0, 1.0)

# Calculate the weighted reflectance using trapezoidal integration
numerator_trapz = np.trapz(I_solar_common * R_interp, x=wl_common_um)
denominator_trapz = np.trapz(I_solar_common, x=wl_common_um)

R_solar_trapz = numerator_trapz / denominator_trapz
R_solar_trapz = round(R_solar_trapz, 6)

# The final calculated value
R_solar_trapz

/tmp/ipykernel_4386/266502826.py:44: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  numerator_trapz = np.trapz(I_solar_common * R_interp, x=wl_common_um)
/tmp/ipykernel_4386/266502826.py:45: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  denominator_trapz = np.trapz(I_solar_common, x=wl_common_um)


np.float64(0.896615)

In [42]:
import numpy as np

# 使用梯形法计算积分
# 对 merged['I_solar'] * merged['R'] 进行积分，分母对 merged['I_solar'] 积分

# 计算分子和分母的积分
numerator_trapz = np.trapezoid(merged['I_solar'] * merged['R'], merged['wavelength_um'])
denominator_trapz = np.trapezoid(merged['I_solar'], merged['wavelength_um'])

R_solar_trapz = numerator_trapz / denominator_trapz
R_solar_trapz = round(R_solar_trapz, 6)
R_solar_trapz

np.float64(0.896615)